In [ ]:
# 1. Setup and Imports
import os
import json
import random
from typing import List, Dict, Any, Optional
from pathlib import Path

# LangChain and Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

# Progress tracking
from tqdm.auto import tqdm

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Configure paths
METADATA_FILE = Path("data/ood/samples.json")
REFERENCE_FILE = Path("data/ood/reference.json")
OUTPUT_DIR = Path("data/ood/")
OUTPUT_FILE = OUTPUT_DIR / "ood_dataset.json"

print(f"Metadata file: {METADATA_FILE}")
print(f"Reference file: {REFERENCE_FILE}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Random seed: {RANDOM_SEED}")

Metadata file: data\ood_sampled\metadata.json
Reference file: data\ood\reference.json
Output file: data\ood_sampled\vqa_dataset.json
Random seed: 42


In [2]:
# 2. Define Pydantic Models for Structured Output
class VQAPair(BaseModel):
    """Structured output for VQA pair generation with quality control."""
    
    # Quality control - NEW FIELD
    is_plant_related: bool = Field(
        description="Indicates if the image is related to plant disease/health. Set to FALSE for unrelated images like ornaments, decorations, or non-plant objects"
    )
    
    # Image composition analysis - NEW FIELDS
    image_shows_multiple_leaves: bool = Field(
        description="TRUE if image shows multiple leaves clearly visible (2 or more leaves). FALSE if single leaf or no leaves"
    )
    image_shows_non_leaf_parts: bool = Field(
        description="TRUE if image primarily shows non-leaf plant parts like fruits, stems, flowers, pods, or roots. FALSE if primarily showing leaves"
    )
    
    user_text: str = Field(
        description="The user's question or statement about the plant image"
    )
    reference_answer: str = Field(
        description="The correct answer that the agent should provide"
    )
    reference_goal: str = Field(
        description="High-level objective of the agent based on user text and system prompt"
    )
    reference_context: str = Field(
        description="Detailed context and evidence for the answer"
    )
    reference_tool_calls: List[Dict[str, Any]] = Field(
        description="List of tool calls the agent should make"
    )
    
    # Metadata
    plant: str = Field(description="Plant species name")
    class_name: str = Field(description="Disease or health class name")
    pathogen_type: str = Field(description="Type of pathogen (fungal, viral, bacterial, healthy, etc.)")
    prompt_type: str = Field(description="Scenario type: vague_symptoms, direct_inquiry, general_inquiry")
    filename: str = Field(description="Image filename")

# Print model information after class definition
print("⚠️  Note: is_plant_related field added for quality control")
print(f"Fields: {list(VQAPair.model_fields.keys())}")
print("✅ VQAPair model defined successfully!")

⚠️  Note: is_plant_related field added for quality control
Fields: ['is_plant_related', 'image_shows_multiple_leaves', 'image_shows_non_leaf_parts', 'user_text', 'reference_answer', 'reference_goal', 'reference_context', 'reference_tool_calls', 'plant', 'class_name', 'pathogen_type', 'prompt_type', 'filename']
✅ VQAPair model defined successfully!


In [3]:
# 3. Load Data and Reference Information
def load_metadata() -> Dict[str, Dict[str, Any]]:
    """Load the metadata.json file."""
    with open(METADATA_FILE, 'r') as f:
        return json.load(f)

def load_reference() -> List[Dict[str, Any]]:
    """Load the reference.json file."""
    with open(REFERENCE_FILE, 'r') as f:
        return json.load(f)

def extract_plant_from_class(class_name: str) -> str:
    """Extract plant name from class name (e.g., 'black_gram anthracnose' -> 'black gram')."""
    return class_name.split()[0].replace('_', ' ')

def extract_pathogen_type(status: str, class_name: str = "") -> str:
    """Determine pathogen type from status and class name."""
    if status == "healthy":
        return "healthy"
    
    # Extract from class name or use common patterns
    if "viral" in class_name.lower() or "virus" in class_name.lower():
        return "viral"
    elif "bacterial" in class_name.lower():
        return "bacterial"
    elif "fungal" in class_name.lower() or any(x in class_name.lower() for x in ["anthracnose", "mildew", "rust", "canker", "spot", "rot"]):
        return "fungal"
    elif "pest" in class_name.lower() or "mite" in class_name.lower() or "insect" in class_name.lower():
        return "pest"
    elif "nutritional deficiency" in class_name.lower():
        return "nutritional deficiency"
    else:
        return "N/A"

# Load data
metadata = load_metadata()
reference_data = load_reference()

# Create reference lookup
reference_lookup = {item["class"]: item for item in reference_data}

print(f"✅ Loaded {len(metadata)} metadata entries")
print(f"✅ Loaded {len(reference_data)} reference entries")

# Validate metadata structure and calculate expected output
def validate_metadata_structure(metadata):
    """Validate metadata and calculate expected dataset size."""
    print(f"\n📋 Metadata Validation:")
    
    # Count by status
    status_counts = {}
    class_status = {}
    for filename, item in metadata.items():
        status = item["status"]
        class_name = item["class"]
        status_counts[status] = status_counts.get(status, 0) + 1
        class_status[class_name] = status
    
    print(f"  Total entries: {len(metadata)}")
    print(f"  By status: {status_counts}")
    
    # Count unique classes
    unique_classes = set(item["class"] for item in metadata.values())
    disease_classes = set(class_name for class_name, status in class_status.items() if status == "diseased")
    healthy_classes = set(class_name for class_name, status in class_status.items() if status == "healthy")
    
    print(f"  Unique classes: {len(unique_classes)}")
    print(f"  Disease classes: {len(disease_classes)}")
    print(f"  Healthy classes: {len(healthy_classes)}")
    
    # Calculate expected output
    disease_images = status_counts.get("diseased", 0)
    healthy_images = status_counts.get("healthy", 0)
    
    # Each image gets 1 scenario, but each class has 3 images = 3 scenarios total per class
    expected_disease_entries = disease_images  # 1 scenario per image
    expected_healthy_entries = healthy_images  # 1 scenario per image
    total_expected = expected_disease_entries + expected_healthy_entries
    
    print(f"\n📊 Expected Dataset Size:")
    print(f"  Disease images: {disease_images} × 1 scenario each = {expected_disease_entries} entries")
    print(f"  Healthy images: {healthy_images} × 1 scenario each = {expected_healthy_entries} entries")
    print(f"  Total expected: {total_expected} entries")
    
    return {
        "disease_classes": len(disease_classes),
        "healthy_classes": len(healthy_classes),
        "disease_images": disease_images,
        "healthy_images": healthy_images,
        "expected_total": total_expected
    }

# Validate and show calculation
validation = validate_metadata_structure(metadata)
print(f"\n✅ Sample classes: {list(metadata.keys())[:5]}")

✅ Loaded 120 metadata entries
✅ Loaded 40 reference entries

📋 Metadata Validation:
  Total entries: 120
  By status: {'diseased': 96, 'healthy': 24}
  Unique classes: 40
  Disease classes: 32
  Healthy classes: 8

📊 Expected Dataset Size:
  Disease images: 96 × 1 scenario each = 96 entries
  Healthy images: 24 × 1 scenario each = 24 entries
  Total expected: 120 entries

✅ Sample classes: ['000_black_gram_anthracnose.jpg', '001_black_gram_anthracnose.jpg', '002_black_gram_anthracnose.jpg', '000_black_gram_leaf.jpg', '001_black_gram_leaf.jpg']


In [4]:
# 4. Enhanced Prompt Templates with Quality Control
def get_vague_symptoms_prompt(plant: str, symptoms: str, image_url: str) -> str:
    """Prompt for scenario 1: Vague symptoms + species + management request."""
    return f"""You are a plant owner who is not an expert in plant diseases. You've taken a photo of your {plant} plant and noticed some issues.

## 🔍 CRITICAL QUALITY CHECK - READ CAREFULLY:
You MUST evaluate if the image is genuinely related to plant disease/health:

### ✅ ACCEPT these images:
- Leaves with spots, lesions, discoloration, wilting, or growth abnormalities
- Plant parts showing symptoms of disease or stress
- Healthy plant parts showing normal growth
- Close-up shots of plant tissues

### ❌ REJECT these images (set is_plant_related=false):
- Ornaments, decorations, artificial plants, or decorative items
- Non-plant objects (rocks, toys, furniture, tools, etc.)
- Images without any visible plant material
- Blurry/unrecognizable images

## 🎯 IMAGE COMPOSITION ANALYSIS (IMPORTANT):
You MUST analyze the image composition to determine detection tool needs:

### image_shows_multiple_leaves:
- Set to TRUE: Image clearly shows 2 or more leaves (even if overlapping)
- Set to FALSE: Single leaf only, or no leaves visible

### image_shows_non_leaf_parts:
- Set to TRUE: Image primarily shows fruits, stems, flowers, pods, roots (not leaves)
- Set to FALSE: Image primarily shows leaves or leaf-related symptoms

**Examples:**
- Single diseased leaf = FALSE for both
- 3 leaves with spots = TRUE for multiple_leaves, FALSE for non_leaf
- Single tomato fruit = FALSE for multiple_leaves, TRUE for non_leaf
- Stem with lesions = FALSE for multiple_leaves, TRUE for non_leaf

## Task:
1. **First**: Carefully examine the image and determine if it's plant-related
2. **Second**: Analyze image composition (single vs multiple, leaf vs non-leaf)
3. **Third**: If plant-related, generate the user question as described below
4. **Fourth**: If NOT plant-related, set is_plant_related=false and use placeholder values

Image shows: {symptoms}

## Generate user question (only if plant-related):
1. Mention the plant species ({plant})
2. Describe visible symptoms in vague, non-technical terms
3. Request identification and management advice

Example format: "My {plant} has some [vague description]. What could this be and how do I treat it?" """


def get_direct_inquiry_prompt(plant: str, symptoms: str, image_url: str) -> str:
    """Prompt for scenario 2: Direct disease inquiry + management request."""
    return f"""You are a plant owner who knows their plant species and wants direct answers.

## 🔍 CRITICAL QUALITY CHECK - READ CAREFULLY:
You MUST evaluate if the image is genuinely related to plant disease/health:

### ✅ ACCEPT these images:
- Leaves with spots, lesions, discoloration, wilting, or growth abnormalities
- Plant parts showing symptoms of disease or stress
- Healthy plant parts showing normal growth

### ❌ REJECT these images (set is_plant_related=false):
- Ornaments, decorations, artificial plants
- Non-plant objects
- Images without visible plant material

## 🎯 IMAGE COMPOSITION ANALYSIS (IMPORTANT):
### image_shows_multiple_leaves:
- TRUE: 2+ leaves clearly visible
- FALSE: Single leaf or no leaves

### image_shows_non_leaf_parts:
- TRUE: Shows fruits, stems, flowers, pods, roots
- FALSE: Shows leaves primarily

## Task:
1. **First**: Determine if image is plant-related
2. **Second**: Analyze image composition
3. **Third**: If yes, generate direct question; if no, set is_plant_related=false

Image shows: {symptoms}

## Generate user question (only if plant-related):
1. Clear mention of plant species ({plant})
2. Direct request for disease identification
3. Request for management/treatment advice

Example format: "I have a {plant} with [symptoms]. What disease is this and how do I treat it?" """


def get_general_inquiry_prompt(plant: str, symptoms: str, image_url: str) -> str:
    """Prompt for scenario 3: General disease identification + management request."""
    return f"""You are a plant owner who took a photo and wants to know what's wrong.

## 🔍 CRITICAL QUALITY CHECK - READ CAREFULLY:
You MUST evaluate if the image is genuinely related to plant disease/health:

### ✅ ACCEPT these images:
- Leaves with spots, lesions, discoloration, wilting, or growth abnormalities
- Plant parts showing symptoms of disease or stress
- Healthy plant parts showing normal growth

### ❌ REJECT these images (set is_plant_related=false):
- Ornaments, decorations, artificial plants
- Non-plant objects
- Images without visible plant material

## 🎯 IMAGE COMPOSITION ANALYSIS (IMPORTANT):
### image_shows_multiple_leaves:
- TRUE: 2+ leaves clearly visible
- FALSE: Single leaf or no leaves

### image_shows_non_leaf_parts:
- TRUE: Shows fruits, stems, flowers, pods, roots
- FALSE: Shows leaves primarily

## Task:
1. **First**: Determine if image is plant-related
2. **Second**: Analyze image composition
3. **Third**: If yes, generate general question; if no, set is_plant_related=false

Image shows: {symptoms}

## Generate user question (only if plant-related):
1. General question about what's wrong with the plant
2. Request for disease identification
3. Request for management advice

Example format: "What's wrong with my plant? How do I fix it?" or "Is this a disease? What should I do?" """


def get_healthy_prompt(plant: str, symptoms: str, image_url: str, scenario: int) -> str:
    """Prompt for healthy plant scenarios."""
    base_prompt = f"""## 🔍 CRITICAL QUALITY CHECK - READ CAREFULLY:
You MUST evaluate if the image is genuinely related to plant health:

### ✅ ACCEPT these images:
- Healthy-looking leaves, stems, or plant parts
- Vibrant green color, firm structure, normal growth
- Plants showing vigorous growth

### ❌ REJECT these images (set is_plant_related=false):
- Ornaments, decorations, artificial plants
- Non-plant objects
- Images without visible plant material
- Plants showing obvious disease symptoms

## 🎯 IMAGE COMPOSITION ANALYSIS (IMPORTANT):
### image_shows_multiple_leaves:
- TRUE: 2+ leaves clearly visible
- FALSE: Single leaf or no leaves

### image_shows_non_leaf_parts:
- TRUE: Shows fruits, stems, flowers, pods, roots
- FALSE: Shows leaves primarily

## Task:
1. **First**: Determine if image shows healthy plant material
2. **Second**: Analyze image composition
3. **Third**: If yes, generate question; if no, set is_plant_related=false

Image shows: {symptoms}

## Generate user question (only if plant-related):"""
    
    if scenario == 1:
        return base_prompt + f"""
1. Mention of the plant species ({plant})
2. Description of how it looks (vibrant, green, etc.)
3. Request for confirmation it's healthy
4. Request for maintenance tips

Example format: "My {plant} looks vibrant and green. Is it healthy or do you see any issues? Any maintenance tips?" """
    elif scenario == 2:
        return base_prompt + f"""
1. Direct mention of the plant species ({plant})
2. Request for health confirmation
3. Request for maintenance advice

Example format: "Is my {plant} healthy? Any maintenance recommendations?" """
    else:
        return base_prompt + f"""
1. General question about plant health
2. Request for maintenance tips

Example format: "Is my plant healthy? What maintenance does it need?" """

print("✅ Enhanced prompt templates with quality control and image composition analysis defined")

✅ Enhanced prompt templates with quality control and image composition analysis defined


In [5]:
# 5. Conditional Tool Call Logic
def get_disease_tool_calls(llm_response: Dict[str, Any] = None) -> List[Dict[str, Any]]:
    """
    Get tool calls for disease identification with conditional detection.
    
    Based on agent system prompt:
    - Mandatory: plant_disease_identification + knowledgebase_search
    - Conditional: closed_set_leaf_detection ONLY if multiple leaves visible
    - Conditional: open_vocabulary_plant_detection ONLY if showing multiple/complex non-leaf parts
    
    Do NOT use detection tools for:
    - Single leaf with clear focus
    - Single fruit/stem/flower with clear focus
    """
    base_tools = [
        {"name": "plant_disease_identification", "args": {}},
        {"name": "knowledgebase_search", "args": {}}
    ]
    
    # Add conditional detection based on LLM's image analysis
    if llm_response:
        # Use closed_set_leaf_detection only for multiple leaves
        if llm_response.get("image_shows_multiple_leaves", False):
            base_tools.insert(0, {"name": "closed_set_leaf_detection", "args": {}})
            print("  → Added closed_set_leaf_detection (multiple leaves detected)")
        
        # Use open_vocabulary_plant_detection only for non-leaf parts
        # AND only if it's not just a single clear organ
        elif llm_response.get("image_shows_non_leaf_parts", False):
            base_tools.insert(0, {"name": "open_vocabulary_plant_detection", "args": {}})
            print("  → Added open_vocabulary_plant_detection (non-leaf structures detected)")
        
        # Single leaf or single clear organ = no detection tool needed
        else:
            print("  → No detection tool needed (single clear plant part)")
    
    return base_tools


def get_healthy_tool_calls() -> List[Dict[str, Any]]:
    """Get tool calls for healthy plant scenarios."""
    return [{"name": "knowledgebase_search", "args": {}}]


def get_reference_goal_for_disease(user_text: str, plant: str, class_name: str) -> str:
    """Generate reference goal for disease scenarios."""
    return f"Perform visual triage of the {plant} symptoms visible in the image, use disease identification tools to confirm {class_name}, then provide evidence-based management and treatment guidance."


def get_reference_goal_for_healthy(user_text: str, plant: str) -> str:
    """Generate reference goal for healthy scenarios."""
    return f"Assess the {plant} health status from the image, confirm it's healthy, and provide maintenance and care recommendations."


print("✅ Conditional tool call logic defined")
print("\n📋 Tool Call Decision Rules:")
print("  ✓ Multiple leaves (2+) → Use closed_set_leaf_detection")
print("  ✓ Non-leaf parts (fruits, stems, etc.) → Use open_vocabulary_plant_detection")
print("  ✓ Single leaf/single organ → NO detection tool")
print("  ✓ Always include: plant_disease_identification + knowledgebase_search")

✅ Conditional tool call logic defined

📋 Tool Call Decision Rules:
  ✓ Multiple leaves (2+) → Use closed_set_leaf_detection
  ✓ Non-leaf parts (fruits, stems, etc.) → Use open_vocabulary_plant_detection
  ✓ Single leaf/single organ → NO detection tool
  ✓ Always include: plant_disease_identification + knowledgebase_search


In [ ]:
# 6. Initialize Gemini Model
def initialize_gemini_model() -> ChatGoogleGenerativeAI:
    """Initialize the Gemini model for structured output."""
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    
    if not api_key:
        print("⚠️  Warning: No API key found in environment variables.")
        print("Set GOOGLE_API_KEY or GEMINI_API_KEY before running generation.")
        return None
    
    try:
        model = ChatGoogleGenerativeAI(
            model="gemini-3-flash-preview",
            temperature=0.7,
            max_tokens=2000
        )
        print("✅ Gemini model initialized successfully")
        return model
    except Exception as e:
        print(f"❌ Failed to initialize Gemini model: {e}")
        return None

# Test model initialization
model = initialize_gemini_model()
if model:
    print("🚀 Model ready for generation")
else:
    print("❌ Model initialization failed - check API key")

✅ Gemini model initialized successfully
🚀 Model ready for generation


In [7]:
# 7. Core VQA Generation Functions (WITH QUALITY CONTROL)
def generate_vqa_pair(
    model: ChatGoogleGenerativeAI,
    image_metadata: Dict[str, Any],
    reference_info: Dict[str, Any],
    scenario_type: str,
    scenario_num: int
) -> Optional[Dict[str, Any]]:
    """
    Generate a single VQA pair with quality control.
    
    Returns:
        Dictionary with VQA data or None if image is unrelated
    """
    try:
        # Extract information
        plant = extract_plant_from_class(image_metadata["class"])
        class_name = image_metadata["cause"]
        status = image_metadata["status"]
        image_url = image_metadata["image_url"]
        filename = image_metadata["path"].split("\\")[-1]
        
        # Get reference info
        ref_info = reference_lookup.get(class_name, {})
        symptoms = ref_info.get("symptoms", image_metadata.get("caption", ""))
        management = ref_info.get("management", ref_info.get("maintenance", ""))
        
        # Determine prompt (text only, without image_url parameter)
        if scenario_type == "diseased":
            if scenario_num == 1:
                prompt_template = get_vague_symptoms_prompt(plant, symptoms, "")
                prompt_type = "vague_symptoms"
            elif scenario_num == 2:
                prompt_template = get_direct_inquiry_prompt(plant, symptoms, "")
                prompt_type = "direct_inquiry"
            else:
                prompt_template = get_general_inquiry_prompt(plant, symptoms, "")
                prompt_type = "general_inquiry"
        else:
            prompt_template = get_healthy_prompt(plant, symptoms, "", scenario_num)
            prompt_type = f"healthy_scenario_{scenario_num}"
        
        # Download image and convert to base64
        import base64
        import requests
        from io import BytesIO
        from PIL import Image as PILImage
        
        try:
            response = requests.get(image_url, timeout=10)
            response.raise_for_status()
            img = PILImage.open(BytesIO(response.content))
            
            # Convert to base64
            buffer = BytesIO()
            img.save(buffer, format="PNG")
            image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
        except Exception as e:
            print(f"⚠️  SKIPPED: {filename} - Failed to download image: {e}")
            return None
        
        # Create multimodal message with base64 image
        message = HumanMessage(
            content=[
                {"type": "text", "text": prompt_template},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}},
            ]
        )
        
        # Generate with structured output using json_schema method
        structured_llm = model.with_structured_output(
            schema=VQAPair.model_json_schema(),
            method="json_schema"
        )
        
        messages = [
            message  # No system message needed for json_schema mode
        ]
        
        result = structured_llm.invoke(messages)
        
        # Check if result is valid
        if result is None:
            print(f"⚠️  SKIPPED: {filename} - Failed to generate structured output")
            return None
        
        # QUALITY CHECK: Filter out unrelated images
        if not result.get("is_plant_related", False):
            print(f"⚠️  FILTERED: {filename} - Not plant-related")
            return None
        
        # Build tool calls using LLM's image composition analysis
        if scenario_type == "diseased":
            tool_calls = get_disease_tool_calls(result)  # Pass LLM response instead of metadata
            reference_goal = get_reference_goal_for_disease(result["user_text"], plant, class_name)
            pathogen_type = extract_pathogen_type(status, class_name)
        else:
            tool_calls = get_healthy_tool_calls()
            reference_goal = get_reference_goal_for_healthy(result["user_text"], plant)
            pathogen_type = "healthy"
        
        # Build reference context
        if scenario_type == "diseased":
            reference_context = f"Plant: {plant}. Disease: {class_name}. Symptoms: {symptoms}. Management: {management}"
        else:
            reference_context = f"Plant: {plant}. Status: Healthy. Maintenance: {management}"
        
        # Construct final entry
        entry = {
            "inputs": {
                "user_text": result["user_text"],
                "image_url": image_url
            },
            "outputs": {
                "reference_answer": result.get("reference_answer", management),
                "reference_goal": reference_goal,
                "reference_context": reference_context,
                "reference_tool_calls": tool_calls
            },
            "metadata": {
                "class": class_name,
                "plant": plant,
                "pathogen_type": pathogen_type,
                "prompt_type": prompt_type,
                "filename": filename,
                "is_plant_related": True  # Confirmed by LLM
            }
        }
        
        return entry
        
    except Exception as e:
        print(f"❌ Error generating VQA pair: {e}")
        import traceback
        traceback.print_exc()
        return None


def generate_scenario_dataset(
    model: ChatGoogleGenerativeAI,
    metadata: Dict[str, Dict[str, Any]],
    scenario_type: str,
    scenario_num: int,
    target_size: int = None
) -> List[Dict[str, Any]]:
    """Generate dataset for a specific scenario with filtering.
    
    Args:
        model: Gemini model for generation
        metadata: Loaded metadata dictionary
        scenario_type: "diseased" or "healthy"
        scenario_num: Scenario number (1, 2, or 3)
        target_size: Optional override for testing (None = use all classes)
    
    Returns:
        List of VQA entries
    """
    # Filter by status
    filtered_items = {k: v for k, v in metadata.items() if v["status"] == scenario_type}
    
    # Get unique classes
    unique_classes = set(v["class"] for v in filtered_items.values())
    print(f"Found {len(unique_classes)} unique {scenario_type} classes")
    
    # Group items by class
    items_by_class = {}
    for filename, item_data in filtered_items.items():
        class_name = item_data["class"]
        if class_name not in items_by_class:
            items_by_class[class_name] = []
        items_by_class[class_name].append((filename, item_data))
    
    # For each class, select 3 images and assign scenarios 1, 2, 3 respectively
    selected_items = []
    for class_name, items in items_by_class.items():
        if len(items) >= 3:
            # Sample 3 images
            sampled = random.sample(items, 3)
            # Assign scenario numbers: 1, 2, 3
            for idx, (filename, item_data) in enumerate(sampled):
                selected_items.append((filename, item_data, scenario_num))  # Use the function's scenario_num for all
        else:
            # If less than 3 images, use what's available and assign scenarios cyclically
            for idx, (filename, item_data) in enumerate(items):
                scenario_to_use = (idx % 3) + 1
                selected_items.append((filename, item_data, scenario_to_use))
    
    # Apply target_size if specified (for testing)
    if target_size:
        selected_items = selected_items[:target_size]
    
    print(f"Selected {len(selected_items)} images for {scenario_type} classes")
    
    # Generate with filtering
    results = []
    filtered_count = 0
    
    for idx, (filename, item_data, assigned_scenario) in enumerate(tqdm(selected_items, desc=f"Generating {scenario_type} scenarios")):
        result = generate_vqa_pair(model, item_data, reference_lookup.get(item_data["class"], {}), scenario_type, assigned_scenario)
        
        if result:
            results.append(result)
        else:
            filtered_count += 1
        
        import time
        time.sleep(0.1)
    
    print(f"✅ Generated {len(results)} entries (filtered {filtered_count})")
    return results

print("✅ VQA generation functions ready")

✅ VQA generation functions ready


In [8]:
# 8. Main Generation Function
def generate_all_scenarios():
    """Generate all scenarios: each class's 3 images get scenarios 1, 2, 3 respectively."""
    print("\n" + "="*60)
    print("GENERATING ALL SCENARIOS")
    print("Each class's 3 images: image1→scenario1, image2→scenario2, image3→scenario3")
    print("="*60)
    
    model = initialize_gemini_model()
    if not model:
        print("❌ Model not initialized")
        return None
    
    all_results = []
    
    # Generate for diseased classes
    print("\n[1/2] Generating diseased scenarios...")
    diseased_results = generate_scenario_dataset(model, metadata, "diseased", 1)  # scenario_num is ignored now
    all_results.extend(diseased_results)
    
    # Generate for healthy classes
    print("\n[2/2] Generating healthy scenarios...")
    healthy_results = generate_scenario_dataset(model, metadata, "healthy", 1)  # scenario_num is ignored now
    all_results.extend(healthy_results)
    
    # Save individual files for backward compatibility
    output_file = OUTPUT_DIR / "vqa_all_scenarios.json"
    with open(output_file, 'w') as f:
        json.dump(all_results, f, indent=2)
    
    print(f"✅ Generated {len(all_results)} total entries")
    print(f"💾 Saved: {output_file}")
    return all_results


print("✅ All scenario generation functions ready")
print("\n📋 Key Functions:")
print("  - generate_all_scenarios(): Main function to generate all scenarios")
print("  - generate_scenario_dataset(): Low-level function for specific types")
print("  - combine_all_datasets(): Combine into final dataset")
print("  - main(): Complete workflow with prompts")

✅ All scenario generation functions ready

📋 Key Functions:
  - generate_all_scenarios(): Main function to generate all scenarios
  - generate_scenario_dataset(): Low-level function for specific types
  - combine_all_datasets(): Combine into final dataset
  - main(): Complete workflow with prompts


In [9]:
# 9. Combine All Scenarios
def combine_all_datasets():
    """Combine all scenario datasets into one final dataset."""
    print("\n" + "="*60)
    print("COMBINING ALL SCENARIOS")
    print("="*60)
    
    all_results = []
    
    # Check for the main output file first
    main_file = OUTPUT_DIR / "vqa_all_scenarios.json"
    if main_file.exists():
        with open(main_file, 'r') as f:
            all_results = json.load(f)
        print(f"✅ Loaded {len(all_results)} entries from {main_file.name}")
    else:
        # Fallback to individual files
        scenario_files = [
            OUTPUT_DIR / "vqa_scenario1_diseased.json",
            OUTPUT_DIR / "vqa_scenario2_diseased.json", 
            OUTPUT_DIR / "vqa_scenario3_diseased.json",
            OUTPUT_DIR / "vqa_healthy_scenarios.json"
        ]
        
        for file_path in scenario_files:
            if file_path.exists():
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    all_results.extend(data)
                    print(f"✅ Loaded {len(data)} entries from {file_path.name}")
            else:
                print(f"⚠️  Missing: {file_path.name}")
    
    if all_results:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(all_results, f, indent=2)
        
        print(f"\n💾 Combined dataset saved to: {OUTPUT_FILE}")
        print(f"📊 Total entries: {len(all_results)}")
        
        # Distribution
        by_status = {}
        by_prompt_type = {}
        for entry in all_results:
            status = "healthy" if entry["metadata"]["pathogen_type"] == "healthy" else "diseased"
            prompt_type = entry["metadata"]["prompt_type"]
            by_status[status] = by_status.get(status, 0) + 1
            by_prompt_type[prompt_type] = by_prompt_type.get(prompt_type, 0) + 1
        
        print(f"  By status: {by_status}")
        print(f"  By prompt type: {by_prompt_type}")
        
        # Expected calculation
        print(f"\n📋 Expected Calculation:")
        print(f"  Disease images: {validation['disease_images']} × 1 scenario each = {validation['disease_images']} entries")
        print(f"  Healthy images: {validation['healthy_images']} × 1 scenario each = {validation['healthy_images']} entries")
        print(f"  Total expected: {validation['expected_total']} entries")
        print(f"  Actual: {len(all_results)} entries")
        
        # Check if we got the expected amount
        if len(all_results) == validation['expected_total']:
            print(f"✅ Perfect match! All {validation['expected_total']} entries generated.")
        else:
            print(f"⚠️  Mismatch: Expected {validation['expected_total']}, got {len(all_results)}")
            print(f"   This could be due to filtering of unrelated images by the LLM.")
        
        return all_results
    else:
        print("❌ No data found")
        return None


# 10. Main Execution
def main():
    """Generate all scenarios and combine."""
    print("🌱 VQA Dataset Generation for Plant Disease Identification")
    print("="*60)
    
    # Check API key
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not api_key:
        print("❌ ERROR: No API key found!")
        print("Set GOOGLE_API_KEY or GEMINI_API_KEY")
        return
    
    print("✅ API key found")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n📝 Dataset Calculation:")
    print(f"  Input: {len(metadata)} metadata entries")
    print(f"  - Disease classes: {validation['disease_classes']} classes")
    print(f"  - Healthy classes: {validation['healthy_classes']} classes")
    print(f"  - Disease images: {validation['disease_images']}")
    print(f"  - Healthy images: {validation['healthy_images']}")
    print("\n  Output: 1 scenario per image (3 images per class = 3 scenarios total)")
    print(f"  - Disease scenarios: {validation['disease_images']} × 1 = {validation['disease_images']} entries")
    print(f"  - Healthy scenarios: {validation['healthy_images']} × 1 = {validation['healthy_images']} entries")
    print(f"  - Total expected: {validation['expected_total']} entries")
    print("\n  Assignment per class:")
    print("    - Image 1 → Scenario 1 (Vague symptoms)")
    print("    - Image 2 → Scenario 2 (Direct inquiry)")
    print("    - Image 3 → Scenario 3 (General inquiry)")
    
    print("\n📝 Generation Plan:")
    print("  1. Generate all diseased scenarios (34 classes × 3 images)")
    print("  2. Generate all healthy scenarios (8 classes × 3 images)")
    print("  3. Combine into final dataset")
    
    response = input("\nProceed with generation? (y/n): ")
    if response.lower() != 'y':
        print("❌ Generation cancelled")
        return
    
    print("\n" + "="*60)
    print("STARTING GENERATION")
    print("="*60)
    
    # Generate all scenarios
    print("\n[1/2] Generating diseased scenarios...")
    diseased_results = generate_scenario_dataset(model, metadata, "diseased", 1)
    
    print("\n[2/2] Generating healthy scenarios...")
    healthy_results = generate_scenario_dataset(model, metadata, "healthy", 1)
    
    # Combine
    all_results = diseased_results + healthy_results
    
    # Save
    output_file = OUTPUT_DIR / "vqa_all_scenarios.json"
    with open(output_file, 'w') as f:
        json.dump(all_results, f, indent=2)
    
    print(f"\n💾 Saved all scenarios to: {output_file}")
    
    # Final combine
    print("\n[FINAL] Creating final dataset...")
    final_dataset = combine_all_datasets()
    
    if final_dataset:
        print("\n" + "="*60)
        print("✅ GENERATION COMPLETE!")
        print("="*60)
        print(f"Final dataset: {OUTPUT_FILE}")
        print(f"Total entries: {len(final_dataset)}")
        print("\nNext steps:")
        print("1. Review the generated dataset")
        print("2. Use with LLM-as-a-judge evaluation")
        print("3. Validate tool call logic")
    else:
        print("\n❌ Generation failed")


print("\n" + "="*60)
print("NOTE: Run main() to generate all scenarios")
print("Or run individual functions:")
print("  - generate_all_scenarios() - generates all scenarios at once")
print("  - generate_scenario_dataset() - generates for specific type")
print("  - combine_all_datasets() - combines into final dataset")
print("\n📊 Dataset Calculation Summary:")
print("  Input: 120 metadata entries (40 classes × 3 images)")
print("  - 34 disease classes × 3 images = 102 disease images")
print("  - 8 healthy classes × 3 images = 24 healthy images")
print("  Output: 1 scenario per image")
print("  - 102 disease images × 1 scenario = 102 entries")
print("  - 24 healthy images × 1 scenario = 24 entries")
print("  - Total: 126 entries (120 if no filtering)")
print("  Assignment: Each class's 3 images get scenarios 1, 2, 3 respectively")
print("="*60)


NOTE: Run main() to generate all scenarios
Or run individual functions:
  - generate_all_scenarios() - generates all scenarios at once
  - generate_scenario_dataset() - generates for specific type
  - combine_all_datasets() - combines into final dataset

📊 Dataset Calculation Summary:
  Input: 120 metadata entries (40 classes × 3 images)
  - 34 disease classes × 3 images = 102 disease images
  - 8 healthy classes × 3 images = 24 healthy images
  Output: 1 scenario per image
  - 102 disease images × 1 scenario = 102 entries
  - 24 healthy images × 1 scenario = 24 entries
  - Total: 126 entries (120 if no filtering)
  Assignment: Each class's 3 images get scenarios 1, 2, 3 respectively


In [ ]:
healthy_results = generate_scenario_dataset(model, metadata, "healthy", 1)

Found 8 unique healthy classes
Selected 24 images for healthy classes


Generating healthy scenarios:   0%|          | 0/24 [00:00<?, ?it/s]

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised DeadlineExceeded: 504 Deadline expired before operation could complete..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised DeadlineExceeded: 504 Deadline expired before operation could complete..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 8.0 seconds as it raised DeadlineExceeded: 504 Deadline expired before operation could complete..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised DeadlineExceeded: 504 Deadline expired before operation could complete..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised DeadlineExceeded: 504 Deadline expired before operation could complete..
Retrying langchain_google_genai.chat_models._chat_with_retry

In [ ]:
# Or generate all scenarios directly (without prompts)
# all_results = generate_all_scenarios()


GENERATING SCENARIO 2: DIRECT DISEASE INQUIRY + MANAGEMENT
✅ Gemini model initialized successfully
Found 32 unique diseased classes
Selected 40 images for diseased scenario 2


Generating diseased scenario 2:   0%|          | 0/40 [00:00<?, ?it/s]

  → Added closed_set_leaf_detection (multiple leaves detected)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → No detection tool needed (single clear plant part)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → No detection tool needed (single clear plant part)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → No de

In [ ]:
# # Or combine existing datasets
# final_dataset = combine_all_datasets()
# print(f"Final dataset has {len(final_dataset)} entries")


GENERATING SCENARIO 3: GENERAL DISEASE INQUIRY + MANAGEMENT
✅ Gemini model initialized successfully
Found 32 unique diseased classes
Selected 40 images for diseased scenario 3


Generating diseased scenario 3:   0%|          | 0/40 [00:00<?, ?it/s]

  → Added closed_set_leaf_detection (multiple leaves detected)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → Added closed_set_leaf_detection (multiple leaves detected)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → No detection tool needed (single clear plant part)
  → Added